# Stage 6 dense full-text embeddings (Private)

Gunakan hanya dengan notebook dan dataset Kaggle **Private**. Output hanya memuat embedding, hash, dan manifest; tidak memuat teks klinis.

In [ ]:
import platform, socket, subprocess, sys
print('python', sys.version)
print('platform', platform.platform())
subprocess.run(['nvidia-smi'], check=True)
socket.create_connection(('huggingface.co', 443), timeout=10).close()
print('GPU_AND_INTERNET_OK')

In [ ]:
%pip install -q --upgrade sentence-transformers==6.1.0 huggingface-hub==1.33.0

In [ ]:
import hashlib, json, time
from pathlib import Path
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
ROOT = Path('/kaggle/input')
matches = sorted(ROOT.rglob('stage6_dense_input.jsonl'))
assert len(matches) == 1, matches
rows = [json.loads(line) for line in matches[0].read_text(encoding='utf-8').splitlines() if line]
assert len(rows) == len({row['sha256'] for row in rows}) == 2205
MODEL_ID = 'intfloat/multilingual-e5-base'
REVISION = 'd128750597153bb5987e10b1c3493a34e5a4502a'
model = SentenceTransformer(MODEL_ID, revision=REVISION, device='cuda')
model.max_seq_length = 512
started = time.perf_counter()
vectors = model.encode(['query: ' + ' '.join(row['text'].split()) for row in rows], batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
elapsed = time.perf_counter() - started
assert vectors.shape == (len(rows), 768) and np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5, rtol=0)
print({'rows': len(rows), 'elapsed_seconds': elapsed, 'gpu': torch.cuda.get_device_name(0)})

In [ ]:
import shutil
OUT = Path('/kaggle/working/stage6_dense_output')
OUT.mkdir(exist_ok=True)
np.save(OUT / 'vectors.npy', vectors)
(OUT / 'hashes.json').write_text(json.dumps([row['sha256'] for row in rows]), encoding='utf-8')
def digest(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
manifest = {'model_id': MODEL_ID, 'revision': REVISION, 'device': torch.cuda.get_device_name(0), 'elapsed_seconds': elapsed, 'rows': len(rows), 'dimension': 768, 'normalized': True, 'sha256': {name: digest(OUT / name) for name in ['vectors.npy', 'hashes.json']}}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/stage6_dense_output', 'zip', OUT)
print(json.dumps(manifest, indent=2))
print('OUTPUT_ARCHIVE', archive)